In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import plotly.graph_objects as go
import math

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

def load_dataset(filename):
    BASE_DIR = './data'
    EXTENSION = '.csv'
    TARGET_PATH = f'{BASE_DIR}/{filename}{EXTENSION}'
    try:
        return pd.read_csv(TARGET_PATH)
    except FileNotFoundError:
        return None
    except UnicodeDecodeError:
        return pd.read_csv(TARGET_PATH, encoding='euc-kr')

df = load_dataset('크루즈입항')

In [3]:
import pandas as pd
import plotly.graph_objects as go

# ============================================================
# 1. 색상 설정
# ============================================================

BLUE = "#3478C0"
ORANGE = "#E67E22"
GRAY = "#657487"


# ============================================================
# 2. 데이터 전처리
# ============================================================

df_month = df.copy()

df_month["입항일시"] = pd.to_datetime(
    df_month["입항일시"],
    errors="coerce"
)

df_month["승객"] = pd.to_numeric(
    df_month["승객"].astype(str).str.replace(",", "", regex=False),
    errors="coerce"
)

df_month = df_month[
    (df_month["입출"].astype(str).str.strip() == "입항") &
    (df_month["선박용도"].astype(str).str.strip() == "크루즈선")
].copy()

df_month = df_month.dropna(
    subset=["입항일시"]
).copy()

df_month = df_month[
    (df_month["입항일시"] >= "2013-01-01") &
    (df_month["입항일시"] < "2026-10-17")
].copy()

df_month["연도"] = df_month["입항일시"].dt.year
df_month["월"] = df_month["입항일시"].dt.month


# ============================================================
# 3. 월별 총승객 수 집계
# ============================================================

monthly = (
    df_month.groupby("월")["승객"]
    .sum(min_count=1)
    .reindex(range(1, 13), fill_value=0)
    .rename("총승객수")
    .reset_index()
)

monthly["월표시"] = monthly["월"].astype(str) + "월"

# 만 명 단위 변환
monthly["총승객수_만명"] = monthly["총승객수"] / 10000


# ============================================================
# 4. 최고 월 설정
# ============================================================

max_month = int(
    monthly.loc[
        monthly["총승객수"].idxmax(), "월"
    ]
)

max_value = monthly["총승객수_만명"].max()

colors = [
    ORANGE if month == max_month else BLUE
    for month in monthly["월"]
]


# ============================================================
# 5. 막대 위 숫자 표시
# ============================================================

labels = [
    f"{value:.1f}" if value > 0 else ""
    for value in monthly["총승객수_만명"]
]


# ============================================================
# 6. 그래프 생성
# ============================================================

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=monthly["월표시"],
        y=monthly["총승객수_만명"],

        marker_color=colors,
        marker_line_width=0,

        text=labels,
        textposition="outside",

        textfont=dict(
            family="Malgun Gothic, Arial, sans-serif",
            size=20,
            color="#243247"
        ),

        cliponaxis=False,

        customdata=monthly[["총승객수"]].to_numpy(),

        hovertemplate=(
            "<b>%{x}</b>"
            "<br>월별 총승객 수: %{customdata[0]:,.0f}명"
            "<extra></extra>"
        )
    )
)


# ============================================================
# 7. 제목 및 레이아웃
# ============================================================

fig.update_layout(
    title=dict(
        text=(
            "<b>부산항 크루즈 월별 총승객 수</b>"
            "<br>"
            f"<span style='font-size:18px;color:{GRAY}'>"
            "월별 승객 규모 및 집중 시기 분석"
            "</span>"
        ),
        x=0.5,
        y=0.95,
        xanchor="center"
    ),

    template="plotly_white",

    width=1200,
    height=675,

    paper_bgcolor="white",
    plot_bgcolor="white",

    font=dict(
        family="Malgun Gothic, Arial, sans-serif",
        size=18,
        color="#243247"
    ),

    margin=dict(
        l=90,
        r=45,
        t=125,
        b=125
    ),

    bargap=0.32,
    showlegend=False
)


# ============================================================
# 8. X축 설정
# ============================================================

fig.update_xaxes(
    title=None,

    type="category",

    categoryorder="array",
    categoryarray=monthly["월표시"].tolist(),

    showgrid=False,

    tickfont=dict(
        family="Malgun Gothic, Arial, sans-serif",
        size=17,
        color="#243247"
    ),

    fixedrange=True
)


# ============================================================
# 9. Y축 설정
# ============================================================

fig.update_yaxes(
    title="총승객 수 (만 명)",

    tickfont=dict(
        family="Malgun Gothic, Arial, sans-serif",
        size=16,
        color="#243247"
    ),

    tickformat=".0f",

    range=[
        0,
        max_value * 1.20
    ],

    gridcolor="#E9EDF2",
    zeroline=False,

    fixedrange=True
)


# ============================================================
# 10. 그래프 하단 주석
# ============================================================

fig.add_annotation(
    x=0,
    y=-0.20,

    xref="paper",
    yref="paper",

    text=(
        "※ 분석 기간: 2013.01.01~2026.10.16"
        "<br>※ 월별 총승객 수는 해당 월의 연도별 승객 수 합계"
        "<br>※ 2026년 10월은 16일까지 집계"
    ),

    showarrow=False,

    xanchor="left",
    align="left",

    font=dict(
        family="Malgun Gothic, Arial, sans-serif",
        size=14,
        color=GRAY
    )
)


# ============================================================
# 11. 그래프 출력 및 이미지 저장 설정
# ============================================================

fig.show(
    config={
        "displaylogo": False,

        "toImageButtonOptions": {
            "format": "png",
            "filename": "PPT_월별_총승객수",
            "width": 1200,
            "height": 675,
            "scale": 3
        }
    }
)